# Specimen Findings — Specimen-Based Measurement Reference

Joins SDTM test identity (what is measured?) with COSMoS dataset specializations
(how is it measured?) for specimen-based Findings domains. Produces a two-sheet
reference file for study design and SoA-to-CDISC mapping.

## Architecture

Two sheets, two levels of resolution:

| Sheet | Level | Primary key | Purpose |
|---|---|---|---|
| Test_Identity | Concept | TESTCD | Step 1: resolve term → what concept? |
| Measurement_Specs | Specification | DS_Code | Step 2: select variant → which specimen/method/scale? |

**Test_Identity** carries every TESTCD from SDTM CT (all domains), enriched with
a COSMoS summary where a Biomedical Concept with specimen-based DSSs exists.
The mapping prompt resolves SoA terms to TESTCDs against this sheet.

**Measurement_Specs** carries Dataset Specializations from specimen-based domains
only (LB, IS, GF, MB, MI, MS, BS, CP, PC, PP, UR). For TESTCDs selected in the
mapping step, the study designer looks up available specifications here — specimen,
method, scale, units, LOINC.

The link between sheets is TESTCD (and NCIt_Code for precision).

## Inputs

| File | Track | Content |
|---|---|---|
| `SDTM_Test_Identity.xlsx` | sdtm-test-codes | 5,781 test codes with NCIt identity |
| `COSMoS_BC_DSS.xlsx` | cosmos-bc-dss | 1,123 dataset specializations (all domains) |
| `SDTM_Domain_Metadata.xlsx` | sdtm-domain-reference | Domain metadata (Specimen_Based flag) |

## Scope

Specimen-based Findings — domains where the pattern is analyte + specimen + method → result.
Identified by `Specimen_Based=Yes` in SDTM_Domain_Metadata.xlsx:

LB (Laboratory), IS (Immunogenicity Specimen), GF (Genomics Findings),
MB (Microbiology Specimen), MI (Microscopic Findings), MS (Microbiology Susceptibility),
BS (Biospecimen Findings), CP (Cell Phenotype), PC (PK Concentrations),
PP (PK Parameters), UR (Urinary System Findings).

## Output

`machine_actionable/Specimen_Findings.xlsx`

## Pipeline Position

```
sdtm-test-codes/SDTM_Test_Identity.xlsx  ──────────┐
cosmos-bc-dss/COSMoS_BC_DSS.xlsx  ─────────────────┤── this notebook
sdtm-domain-reference/SDTM_Domain_Metadata.xlsx  ──┘
                                                     │
                        sdtm-findings/machine_actionable/Specimen_Findings.xlsx
```

In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from collections import Counter
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print('SPECIMEN FINDINGS — SPECIMEN-BASED MEASUREMENT REFERENCE')
print(f'Run: {datetime.now():%Y-%m-%d %H:%M}')

SPECIMEN FINDINGS — SPECIMEN-BASED MEASUREMENT REFERENCE
Run: 2026-03-13 14:19


## 1. Setup

In [2]:
BASE_DIR = Path.cwd().parent        # sdtm-findings/
REPO_ROOT = BASE_DIR.parent          # cdisc-for-ai/

GREEN_FILE = REPO_ROOT / 'sdtm-test-codes' / 'machine_actionable' / 'SDTM_Test_Identity.xlsx'
YELLOW_FILE = REPO_ROOT / 'cosmos-bc-dss' / 'interim' / 'COSMoS_BC_DSS.xlsx'
DOMAIN_META_FILE = REPO_ROOT / 'sdtm-domain-reference' / 'machine_actionable' / 'SDTM_Domain_Metadata.xlsx'

MA_DIR = BASE_DIR / 'machine_actionable'
MA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = MA_DIR / 'Specimen_Findings.xlsx'

for f, label in [(GREEN_FILE, 'Green'), (YELLOW_FILE, 'Yellow'), (DOMAIN_META_FILE, 'Domain Meta')]:
    if f.exists():
        print(f'  {label}: {f}')
    else:
        raise FileNotFoundError(f'{label} file not found: {f}')

print(f'  Output: {OUTPUT_FILE}')

  Green: /Users/kerstinforsberg/repos/GitHub/cdisc-for-ai/sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx
  Yellow: /Users/kerstinforsberg/repos/GitHub/cdisc-for-ai/cosmos-bc-dss/interim/COSMoS_BC_DSS.xlsx
  Domain Meta: /Users/kerstinforsberg/repos/GitHub/cdisc-for-ai/sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx
  Output: /Users/kerstinforsberg/repos/GitHub/cdisc-for-ai/sdtm-findings/machine_actionable/Specimen_Findings.xlsx


## 2. Load Input Files

In [3]:
green = pd.read_excel(GREEN_FILE, sheet_name='Test Codes', dtype=str).fillna('')
yellow = pd.read_excel(YELLOW_FILE, sheet_name='BC_DSS', dtype=str).fillna('')
domain_meta = pd.read_excel(DOMAIN_META_FILE, sheet_name='Domains', dtype=str)

print(f'Green: {len(green):,} rows x {len(green.columns)} columns')
print(f'Yellow: {len(yellow):,} rows x {len(yellow.columns)} columns')
print(f'Yellow row types: {yellow["Row_Type"].value_counts().to_dict()}')
print(f'Domain metadata: {len(domain_meta)} domains')

Green: 5,781 rows x 9 columns
Yellow: 1,545 rows x 34 columns
Yellow row types: {'DSS': 1123, 'BC_only_hierarchy': 228, 'BC_only_no_DS': 194}
Domain metadata: 56 domains


## 2b. Identify Specimen-Based Domains

Use SDTM_Domain_Metadata.xlsx to identify which domains are specimen-based.
This drives the Measurement_Specs scope and the COSMoS summary enrichment.

In [4]:
specimen_domains = set(
    domain_meta[domain_meta['Specimen_Based'] == 'Yes']['Domain']
)
print(f'Specimen-based domains ({len(specimen_domains)}): {sorted(specimen_domains)}')

Specimen-based domains (11): ['BS', 'CP', 'GF', 'IS', 'LB', 'MB', 'MI', 'MS', 'PC', 'PP', 'UR']


## 3. Build Sheet 1: Test_Identity

One row per TESTCD. Full green baseline (all domains) enriched with COSMoS summary
scoped to specimen-based DSSs.

COSMoS enrichment per TESTCD:
- `Has_COSMoS` — whether a COSMoS BC exists with specimen-based DSSs for this test code
- `COSMoS_DSS_Count` — number of specimen-based dataset specializations
- `COSMoS_BC_Name` — BC name(s) from COSMoS
- `COSMoS_Categories` — BC categories (for discovery/grouping)
- `COSMoS_Hierarchy` — hierarchy path (for panel/group context)
- `COSMoS_Domains` — specimen-based domains that have DSSs for this BC

In [5]:
# Extract COSMoS summary per TESTCD from DSS rows — specimen domains only
dss_all = yellow[yellow['Row_Type'] == 'DSS'].copy()
dss_specimen = dss_all[
    (dss_all['TESTCD'] != '') &
    (dss_all['Domain'].isin(specimen_domains))
].copy()

print(f'All DSS rows: {len(dss_all):,}')
print(f'Specimen-domain DSS rows with TESTCD: {len(dss_specimen):,}')
print(f'Domains in scope: {sorted(dss_specimen["Domain"].unique())}')


def cosmos_summary_for_testcd(group):
    """Summarize COSMoS BC/DSS info for one TESTCD (specimen domains only)."""
    bc_names = group['BC_Name'].unique()
    categories = group['Categories'].unique()
    hierarchies = group['Hierarchy_Path'].unique()
    domains = sorted(group['Domain'].unique())

    return pd.Series({
        'Has_COSMoS': 'Yes',
        'COSMoS_DSS_Count': str(len(group)),
        'COSMoS_BC_Name': '; '.join(n for n in bc_names if n),
        'COSMoS_Categories': '; '.join(c for c in categories if c),
        'COSMoS_Hierarchy': '; '.join(h for h in hierarchies if h),
        'COSMoS_Domains': '; '.join(domains),
    })

cosmos_summary = dss_specimen.groupby('TESTCD').apply(
    cosmos_summary_for_testcd
).reset_index()

print(f'COSMoS summary: {len(cosmos_summary):,} TESTCDs with specimen-based DSS coverage')

All DSS rows: 1,123
Specimen-domain DSS rows with TESTCD: 489
Domains in scope: ['GF', 'IS', 'LB', 'MB', 'MI', 'UR']
COSMoS summary: 127 TESTCDs with specimen-based DSS coverage


In [6]:
# Merge green baseline with COSMoS summary
test_codes = green.merge(
    cosmos_summary,
    on='TESTCD',
    how='left'
).fillna('')

# Fill Has_COSMoS for unmatched rows
test_codes.loc[test_codes['Has_COSMoS'] == '', 'Has_COSMoS'] = 'No'
test_codes.loc[test_codes['COSMoS_DSS_Count'] == '', 'COSMoS_DSS_Count'] = '0'

n_with = (test_codes['Has_COSMoS'] == 'Yes').sum()
n_without = (test_codes['Has_COSMoS'] == 'No').sum()
print(f'Test_Identity: {len(test_codes):,} rows')
print(f'  With COSMoS (specimen): {n_with:,} ({n_with/len(test_codes)*100:.1f}%)')
print(f'  Without COSMoS: {n_without:,} ({n_without/len(test_codes)*100:.1f}%)')

Test_Identity: 5,781 rows
  With COSMoS (specimen): 127 (2.2%)
  Without COSMoS: 5,654 (97.8%)


## 4. Build Sheet 2: Measurement_Specs

One row per Dataset Specialization from specimen-based domains only.
Carries the TESTCD for linking back to Sheet 1.
Includes only DSS rows (operational specifications), not hierarchy or BC-only rows.

In [7]:
SPEC_COLS = [
    # Link to Sheet 1
    'TESTCD', 'TEST', 'NCIt_Code',
    # BC identity
    'COSMoS_BC_ID', 'BC_Name', 'BC_Type',
    # DSS identity
    'DS_Code', 'DS_Name', 'Domain', 'Domain_Class',
    # Measurement specification
    'Result_Scale', 'Specimen', 'Specimen_NCIt', 'Method', 'Method_NCIt',
    # Reference codes
    'LOINC_Code_Count', 'LOINC_Code', 'Allowed_Units', 'Standard_Unit',
    # Contextual
    'location', 'laterality', 'evaluator',
]

# Specimen-domain DSS rows only
specs = dss_all[dss_all['Domain'].isin(specimen_domains)][SPEC_COLS].copy()

print(f'Measurement_Specs: {len(specs):,} rows')
print(f'  With TESTCD: {(specs["TESTCD"] != "").sum():,}')
print(f'  Without TESTCD: {(specs["TESTCD"] == "").sum():,}')
print(f'  Domains: {specs["Domain"].value_counts().to_dict()}')
print(f'  With specimen: {(specs["Specimen"] != "").sum():,}')
print(f'  With LOINC: {(specs["LOINC_Code"] != "").sum():,}')

Measurement_Specs: 494 rows
  With TESTCD: 489
  Without TESTCD: 5
  Domains: {'IS': 290, 'LB': 142, 'GF': 38, 'UR': 10, 'MI': 7, 'MB': 7}
  With specimen: 485
  With LOINC: 138


## 5. Coverage Analysis

How much of the green file is enriched by specimen-based COSMoS data?

In [8]:
print('=== Coverage ===')
print(f'Green TESTCDs:         {len(test_codes):,}')
print(f'  With COSMoS (specimen): {n_with:,} ({n_with/len(test_codes)*100:.1f}%)')
print(f'  Without COSMoS:      {n_without:,} ({n_without/len(test_codes)*100:.1f}%)')
print()

# Domain breakdown for COSMoS coverage
with_cosmos = test_codes[test_codes['Has_COSMoS'] == 'Yes']
print('COSMoS specimen coverage by green SDTM_Domains:')
domain_coverage = Counter()
domain_total = Counter()
for _, row in test_codes.iterrows():
    for d in str(row['SDTM_Domains']).split('; '):
        if d.strip():
            domain_total[d.strip()] += 1
            if row['Has_COSMoS'] == 'Yes':
                domain_coverage[d.strip()] += 1

for dom in sorted(domain_total, key=domain_total.get, reverse=True)[:15]:
    cov = domain_coverage.get(dom, 0)
    tot = domain_total[dom]
    pct = cov / tot * 100 if tot else 0
    marker = ' *' if dom in specimen_domains else ''
    print(f'  {dom:6s}  {cov:4d} / {tot:4d}  ({pct:5.1f}%){marker}')

print()
print('  * = specimen-based domain')

=== Coverage ===
Green TESTCDs:         5,781
  With COSMoS (specimen): 127 (2.2%)
  Without COSMoS:      5,654 (97.8%)

COSMoS specimen coverage by green SDTM_Domains:
  LB        95 / 2474  (  3.8%) *
  CP        10 /  930  (  1.1%) *
  MB         5 /  555  (  0.9%) *
  FA         0 /  228  (  0.0%)
  MI         7 /  222  (  3.2%) *
  PT         5 /  208  (  2.4%)
  CV         2 /  191  (  1.0%)
  RE         2 /  136  (  1.5%)
  EG         0 /  111  (  0.0%)
  RP         0 /  105  (  0.0%)
  IS         7 /   99  (  7.1%) *
  NV         1 /   87  (  1.1%)
  SC         0 /   79  (  0.0%)
  TR         0 /   76  (  0.0%)
  VS         0 /   75  (  0.0%)

  * = specimen-based domain


## 6. Write Output

In [9]:
# Column definitions for Sheet 1
TESTCODE_COLS = [
    # Green identity
    ('NCIt_Code',           12),
    ('TESTCD',              12),
    ('TEST',                38),
    ('SDTM_Domains',        18),
    ('NCIt_Preferred_Term', 45),
    ('NCIt_Synonyms',       55),
    ('NCIt_Definition',     60),
    ('UMLS_CUI',            14),
    ('NCIm_CUI',            14),
    # COSMoS summary (specimen scope)
    ('Has_COSMoS',          10),
    ('COSMoS_DSS_Count',    10),
    ('COSMoS_BC_Name',      40),
    ('COSMoS_Categories',   30),
    ('COSMoS_Hierarchy',    40),
    ('COSMoS_Domains',      16),
]

# Column definitions for Sheet 2
SPEC_COL_WIDTHS = [
    ('TESTCD',              12),
    ('TEST',                38),
    ('NCIt_Code',           12),
    ('COSMoS_BC_ID',        14),
    ('BC_Name',             35),
    ('BC_Type',             10),
    ('DS_Code',             14),
    ('DS_Name',             38),
    ('Domain',              8),
    ('Domain_Class',        14),
    ('Result_Scale',        14),
    ('Specimen',            22),
    ('Specimen_NCIt',       12),
    ('Method',              18),
    ('Method_NCIt',         12),
    ('LOINC_Code_Count',    10),
    ('LOINC_Code',          18),
    ('Allowed_Units',       25),
    ('Standard_Unit',       14),
    ('location',            20),
    ('laterality',          14),
    ('evaluator',           14),
]

print('Column definitions ready.')

Column definitions ready.


In [10]:
# ── Styles — green/yellow colour scheme matching upstream notebooks ──
HEADER_FONT = Font(name='Arial', bold=True, size=10, color='FFFFFF')
DATA_FONT = Font(name='Arial', size=10)
WRAP = Alignment(wrap_text=True, vertical='top')
THIN_BORDER = Border(bottom=Side(style='thin', color='D0D0D0'))

# Header fills — from upstream notebooks
GREEN_HEADER = PatternFill('solid', fgColor='548235')   # Enrich notebook: HDR_FILL
YELLOW_HEADER = PatternFill('solid', fgColor='FFD700')  # Flatten notebook: YELLOW_HEADER

# Data fills
COSMOS_YES = PatternFill('solid', fgColor='FFFCE8')     # Flatten notebook: YELLOW_FILL
COSMOS_NO = PatternFill('solid', fgColor='F2F2F2')      # grey


def write_sheet(ws, df, col_defs, header_fill=None):
    """Write dataframe to worksheet with formatting."""
    col_names = [c[0] for c in col_defs]
    fill = header_fill or GREEN_HEADER

    # Headers
    for ci, name in enumerate(col_names, 1):
        cell = ws.cell(row=1, column=ci, value=name)
        cell.font = HEADER_FONT
        cell.fill = fill
        cell.alignment = WRAP

    # Data
    for ri, (_, row) in enumerate(df.iterrows(), 2):
        for ci, name in enumerate(col_names, 1):
            val = row.get(name, '')
            cell = ws.cell(row=ri, column=ci, value=val if val != '' else None)
            cell.font = DATA_FONT
            cell.alignment = WRAP

    # Column widths
    for ci, (name, width) in enumerate(col_defs, 1):
        ws.column_dimensions[get_column_letter(ci)].width = width

    # Freeze and filter
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = f'A1:{get_column_letter(len(col_names))}1'


print('Writer functions ready.')

Writer functions ready.


In [11]:
wb = Workbook()

# ── README ──
ws_readme = wb.active
ws_readme.title = 'README'

readme_font = Font(name='Arial', size=10)
title_font = Font(name='Arial', size=12, bold=True)
section_font = Font(name='Arial', size=10, bold=True)

readme_lines = [
    ('Specimen Findings — Specimen-Based Measurement Reference', title_font),
    ('', None),
    ('PROVENANCE', section_font),
    (f'Generated: {datetime.now():%Y-%m-%d %H:%M}', readme_font),
    (f'Green source: {GREEN_FILE.name}', readme_font),
    (f'Yellow source: {YELLOW_FILE.name}', readme_font),
    (f'Domain metadata: {DOMAIN_META_FILE.name}', readme_font),
    ('Notebook: Specimen_Findings.ipynb (sdtm-findings track)', readme_font),
    ('', None),
    ('SCOPE', section_font),
    (f'Test_Identity: all {len(test_codes):,} TESTCDs (full green baseline, all domains)', readme_font),
    (f'Measurement_Specs: {len(specs):,} DSS rows from specimen-based domains only', readme_font),
    (f'Specimen domains: {" ".join(sorted(specimen_domains))}', readme_font),
    ('', None),
    ('SHEETS', section_font),
    ('Test_Identity — one row per TESTCD. Green identity + COSMoS specimen summary.', readme_font),
    ('  The mapping prompt resolves SoA terms against this sheet.', readme_font),
    ('Measurement_Specs — one row per Dataset Specialization (specimen domains only).', readme_font),
    ('  For selected TESTCDs, look up specimen/method/scale/units/LOINC here.', readme_font),
    ('', None),
    ('TEST_IDENTITY COLUMNS (green identity)', section_font),
    ('NCIt_Code — NCIt C-code for the test concept', readme_font),
    ('TESTCD — SDTM short test code (submission value)', readme_font),
    ('TEST — SDTM full test name (submission value)', readme_font),
    ('SDTM_Domains — domain memberships (semicolon-separated)', readme_font),
    ('NCIt_Preferred_Term — preferred term from SDTM Terminology', readme_font),
    ('NCIt_Synonyms — alternative names from NCIt (semicolon-separated)', readme_font),
    ('NCIt_Definition — definition from NCIt Thesaurus', readme_font),
    ('UMLS_CUI — Standard UMLS CUI for crosslinking (SNOMED, MeSH, etc.)', readme_font),
    ('NCIm_CUI — NCI Metathesaurus CUI (NCI-internal)', readme_font),
    ('', None),
    ('TEST_IDENTITY COLUMNS (COSMoS specimen summary)', section_font),
    ('Has_COSMoS — Yes if a COSMoS BC with specimen-based DSS exists for this TESTCD', readme_font),
    ('COSMoS_DSS_Count — number of specimen-based dataset specializations', readme_font),
    ('COSMoS_BC_Name — biomedical concept name(s) from COSMoS', readme_font),
    ('COSMoS_Categories — BC categories for discovery and grouping', readme_font),
    ('COSMoS_Hierarchy — hierarchy path for panel/group context', readme_font),
    ('COSMoS_Domains — specimen-based domains with dataset specializations', readme_font),
    ('', None),
    ('MEASUREMENT_SPECS COLUMNS', section_font),
    ('TESTCD, TEST, NCIt_Code — link to Test_Identity sheet', readme_font),
    ('COSMoS_BC_ID — COSMoS internal BC identifier', readme_font),
    ('BC_Name — biomedical concept name', readme_font),
    ('DS_Code — dataset specialization mnemonic code (COSMoS vlm_group_id)', readme_font),
    ('DS_Name — dataset specialization name', readme_font),
    ('Domain, Domain_Class — SDTM domain and observation class', readme_font),
    ('Result_Scale — Quantitative / Qualitative / etc.', readme_font),
    ('Specimen, Method — resolved SDTM CT submission values', readme_font),
    ('LOINC_Code — LOINC code(s) where available', readme_font),
    ('Allowed_Units, Standard_Unit — unit specifications', readme_font),
    ('location, laterality, evaluator — contextual qualifiers (sparse)', readme_font),
    ('', None),
    ('COVERAGE', section_font),
    (f'Total TESTCDs: {len(test_codes):,}', readme_font),
    (f'With COSMoS specimen DSS: {n_with:,} ({n_with/len(test_codes)*100:.1f}%)', readme_font),
    (f'Without COSMoS: {n_without:,} ({n_without/len(test_codes)*100:.1f}%)', readme_font),
    (f'Specimen-based DSS rows: {len(specs):,}', readme_font),
    ('', None),
    ('STATUS', section_font),
    ('Consumer track output. Column names use COSMoS vocabulary for traceability.', readme_font),
    ('Not an official CDISC product. Sources: NCI EVS + COSMoS public exports.', readme_font),
]

for ri, (text, font) in enumerate(readme_lines, 1):
    cell = ws_readme.cell(row=ri, column=1, value=text if text else None)
    if font:
        cell.font = font

ws_readme.column_dimensions['A'].width = 100
print(f'README: {len(readme_lines)} lines')

README: 61 lines


In [12]:
# ── Test_Identity sheet ──
ws_tc = wb.create_sheet('Test_Identity')
tc_col_names = [c[0] for c in TESTCODE_COLS]
write_sheet(ws_tc, test_codes[tc_col_names], TESTCODE_COLS, header_fill=GREEN_HEADER)

# Override COSMoS summary columns (last 6) with yellow headers
cosmos_start = len(TESTCODE_COLS) - 6 + 1  # 1-indexed
for ci in range(cosmos_start, len(TESTCODE_COLS) + 1):
    ws_tc.cell(row=1, column=ci).fill = YELLOW_HEADER

# Conditional fill on Has_COSMoS column
has_cosmos_ci = tc_col_names.index('Has_COSMoS') + 1
for ri in range(2, len(test_codes) + 2):
    val = ws_tc.cell(row=ri, column=has_cosmos_ci).value
    if val == 'Yes':
        ws_tc.cell(row=ri, column=has_cosmos_ci).fill = COSMOS_YES
    elif val == 'No':
        ws_tc.cell(row=ri, column=has_cosmos_ci).fill = COSMOS_NO

print(f'Test_Identity: {len(test_codes):,} rows')

Test_Identity: 5,781 rows


In [13]:
# ── Measurement_Specs sheet ──
ws_specs = wb.create_sheet('Measurement_Specs')
spec_col_names = [c[0] for c in SPEC_COL_WIDTHS]
write_sheet(ws_specs, specs[spec_col_names], SPEC_COL_WIDTHS, header_fill=YELLOW_HEADER)

# Link columns (TESTCD, TEST, NCIt_Code) get green headers — they're the bridge
for ci in range(1, 4):  # first 3 columns
    ws_specs.cell(row=1, column=ci).fill = GREEN_HEADER

print(f'Measurement_Specs: {len(specs):,} rows')

Measurement_Specs: 494 rows


## 7. Save and Summary

In [14]:
wb.save(OUTPUT_FILE)
print(f'Output: {OUTPUT_FILE}')
print(f'File size: {OUTPUT_FILE.stat().st_size / 1024:.0f} KB')
print()
print('=== Summary ===')
print(f'Test_Identity sheet:      {len(test_codes):,} rows  (one per TESTCD, all domains)')
print(f'  With COSMoS (specimen): {n_with:,}')
print(f'  Without COSMoS:         {n_without:,}')
print(f'Measurement_Specs sheet:  {len(specs):,} rows  (specimen domains only)')
print(f'  With specimen:          {(specs["Specimen"] != "").sum():,}')
print(f'  With LOINC:             {(specs["LOINC_Code"] != "").sum():,}')
print(f'  Domains:                {sorted(specs["Domain"].unique())}')
print()
print('Reference file ready for mapping prompt.')

Output: /Users/kerstinforsberg/repos/GitHub/cdisc-for-ai/sdtm-findings/machine_actionable/Specimen_Findings.xlsx
File size: 671 KB

=== Summary ===
Test_Identity sheet:      5,781 rows  (one per TESTCD, all domains)
  With COSMoS (specimen): 127
  Without COSMoS:         5,654
Measurement_Specs sheet:  494 rows  (specimen domains only)
  With specimen:          485
  With LOINC:             138
  Domains:                ['GF', 'IS', 'LB', 'MB', 'MI', 'UR']

Reference file ready for mapping prompt.
